# DBSCAN in JAX

This notebook demonstrates density-based clustering (DBSCAN). The implementation uses NumPy for the region queries for simplicity while data creation and visualization use JAX arrays.

DBSCAN groups together points that are closely packed while marking low-density points as noise.

## Theory

DBSCAN relies on two parameters: radius $arepsilon$ and minimum samples $	ext{minPts}$. A point $p$ is a core point if the number of points within $arepsilon$ of $p$ is at least $	ext{minPts}$. Points within $arepsilon$ of a core point belong to the same cluster.

Distance used is Euclidean: $d(x_i, x_j) = x_i - x_j_2$.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as onp
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(21)
centers = jnp.array([[-4.0, -2.0], [0.0, 4.0], [4.0, -1.0]])
samples = []
for i, c in enumerate(centers):
    key, sub = jax.random.split(key)
    samples.append(jax.random.normal(sub, (100, 2)) * 0.6 + c)
noise_key = jax.random.split(key, 1)[0]
noise = jax.random.uniform(noise_key, (30, 2), minval=-6.0, maxval=6.0)
X = jnp.concatenate(samples + [noise], axis=0)
X = X[jax.random.permutation(key, X.shape[0])]

In [ ]:
def dbscan(X, eps=0.8, min_samples=5):
    Xnp = onp.array(X)
    n = Xnp.shape[0]
    labels = -onp.ones(n, dtype=int)
    visited = onp.zeros(n, dtype=bool)
    cluster_id = 0
    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        distances = onp.linalg.norm(Xnp - Xnp[i], axis=1)
        neighbors = list(onp.where(distances <= eps)[0])
        if len(neighbors) < min_samples:
            labels[i] = -1
        else:
            labels[neighbors] = cluster_id
            queue = neighbors[:]
            while queue:
                j = queue.pop(0)
                if not visited[j]:
                    visited[j] = True
                    d2 = onp.linalg.norm(Xnp - Xnp[j], axis=1)
                    neigh2 = list(onp.where(d2 <= eps)[0])
                    if len(neigh2) >= min_samples:
                        for nb in neigh2:
                            if labels[nb] == -1:
                                labels[nb] = cluster_id
                            if labels[nb] == -1 or labels[nb] != cluster_id:
                                pass
                            if labels[nb] == -1:
                                queue.append(nb)
                if labels[j] == -1:
                    labels[j] = cluster_id
            cluster_id += 1
    return labels

labels = dbscan(X, eps=0.9, min_samples=6)
labels

## Results

Visualize clustered points where label -1 denotes noise.

In [ ]:
unique_labels = onp.unique(labels)
colors = [plt.cm.tab10(i % 10) if lab != -1 else (0.0, 0.0, 0.0, 1.0) for i, lab in enumerate(unique_labels)]
color_map = {lab: colors[i] for i, lab in enumerate(unique_labels)}
point_colors = [color_map[l] for l in labels]
plt.figure(figsize=(7,6))
plt.scatter(X[:,0], X[:,1], c=point_colors, s=30, alpha=0.85)
plt.title('DBSCAN Clustering (JAX data, NumPy DBSCAN)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()